# Baseline Inference — mBERT + Bi-LSTM + CRF (model #13)

Notebook này nạp checkpoint đã train của **mBERT + Bi-LSTM + CRF (model #13 trong
model_tracking.xlsx, backbone `bert-base-multilingual-cased`)** từ `baseline_lstm_crf.py`,
chạy trên **GPU local**, và suy luận (thêm dấu câu) trên câu tiếng Việt chưa có dấu câu.

**Yêu cầu đầu vào (giống các notebook trước):**
- Câu tiếng Việt **có dấu**, không romanized.
- Các từ cách nhau bởi khoảng trắng, đúng cách tách từ như lúc train.
- `max_seq_length=256` (subword) — câu quá dài sẽ bị cắt, notebook sẽ cảnh báo.

**Trước khi chạy:** sửa các biến trong ô "CONFIG" bên dưới. Notebook này mặc định trỏ tới
checkpoint **Novels** (đã train xong) — đổi sang News khi có checkpoint đó sau này.

## 1. Cấu hình đường dẫn + model

In [3]:
import sys
import os

# --------------------------------------------------------------------- #
# CONFIG - SỬA CÁC DÒNG DƯỚI ĐÂY CHO KHỚP VỚI MÁY FA
# --------------------------------------------------------------------- #

# Thư mục chứa baseline_lstm_crf.py + punc_dataset_word.py (script train gốc)
SCRIPT_DIR = r"D:\COLING2027\2026_08_09\phopunct"

# Checkpoint đã tải về máy (Novels đã train xong; đổi sang News khi có sau này)
CHECKPOINT_PATH = r"D:\COLING2027\2026_08_09\phopunct\outputs_from_gpu\mbert_bilstm_news\best_checkpoint.pt"
# CHECKPOINT_PATH = r"D:\COLING2027\2026_08_09\phopunct\outputs_from_gpu\mbert_bilstm_news\best_checkpoint.pt"

# Model key + use_bilstm PHẢI khớp đúng lúc train (model #13 = mbert + bilstm)
MODEL_KEY = "mbert"       # mbert | velectra | bert | xlmr
USE_BILSTM = True         # True cho model #13-16, False cho model #9-12

# GPU local - đổi "cpu" nếu máy không có CUDA khả dụng
DEVICE = "cpu"

MAX_SEQ_LENGTH = 256
LSTM_HIDDEN_SIZE = 128    # phải khớp lúc train (mặc định trong baseline_lstm_crf.py)

# --------------------------------------------------------------------- #

sys.path.insert(0, SCRIPT_DIR)
assert os.path.isfile(CHECKPOINT_PATH), f"Không tìm thấy checkpoint: {CHECKPOINT_PATH}"

import torch
if DEVICE == "cuda" and not torch.cuda.is_available():
    print("[Cảnh báo] Không phát hiện CUDA khả dụng trên máy này, chuyển tạm sang CPU.")
    DEVICE = "cpu"

print(f"OK - sys.path, checkpoint sẵn sàng. DEVICE={DEVICE}, torch={torch.__version__}, "
      f"cuda_available={torch.cuda.is_available()}")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


OK - sys.path, checkpoint sẵn sàng. DEVICE=cpu, torch=2.11.0+cpu, cuda_available=False


## 2. Nạp tokenizer + model từ checkpoint

In [4]:
from baseline_lstm_crf import TransformerWordCrf, BACKBONES, MODEL_DISPLAY_NAME
from punc_dataset_word import LABELS, ID2LABEL, LABEL2ID
from transformers import AutoTokenizer, BertTokenizer

device = torch.device(DEVICE)
bert_model_name = BACKBONES[MODEL_KEY]
display_name = MODEL_DISPLAY_NAME[(MODEL_KEY, USE_BILSTM)]
print(f"Đang nạp: {display_name} | backbone={bert_model_name}")

# mBERT (bert-base-multilingual-cased) là checkpoint chuẩn Google, dùng BertTokenizer/BertModel
# trực tiếp - đồng bộ với cách baseline_lstm_crf.py nạp lúc train (không dính lỗi model_type
# như checkpoint BERT cộng đồng cũ NlpHUST/vibert4news-base-cased).
print("Đang nạp tokenizer (lần đầu có thể mất thời gian tải về nếu chưa có cache)...")
if MODEL_KEY in ("mbert", "bert"):
    tokenizer = BertTokenizer.from_pretrained(bert_model_name)
else:
    tokenizer = AutoTokenizer.from_pretrained(bert_model_name, use_fast=False)

print("Đang khởi tạo model + nạp checkpoint...")
model = TransformerWordCrf(
    bert_model_name, num_labels=len(LABELS),
    use_bilstm=USE_BILSTM, lstm_hidden_size=LSTM_HIDDEN_SIZE,
    use_bert_encoder=(MODEL_KEY in ("mbert", "bert")),
).to(device)

ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

ckpt_display = ckpt.get("display_name", display_name)
print(f"Nạp xong [{ckpt_display}]. Checkpoint từ epoch {ckpt.get('epoch')}, "
      f"best_f1 lúc lưu = {ckpt.get('best_f1'):.4f}")

if "model_key" in ckpt and ckpt["model_key"] != MODEL_KEY:
    print(f"[Cảnh báo] Checkpoint lưu model_key='{ckpt['model_key']}' khác với MODEL_KEY='{MODEL_KEY}' đang chọn!")
if "use_bilstm" in ckpt and ckpt["use_bilstm"] != USE_BILSTM:
    print(f"[Cảnh báo] Checkpoint lưu use_bilstm={ckpt['use_bilstm']} khác với USE_BILSTM={USE_BILSTM} đang chọn!")


c:\Users\Dell\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Đang nạp: mBERT + Bi-LSTM + CRF | backbone=bert-base-multilingual-cased
Đang nạp tokenizer (lần đầu có thể mất thời gian tải về nếu chưa có cache)...


08/11/2026 19:51:32 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/bert-base-multilingual-cased/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
08/11/2026 19:51:32 - WARNING - huggingface_hub.utils._http - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
08/11/2026 19:51:33 - INFO - httpx - HTTP Request: GET https://huggingface.co/bert-base-multilingual-cased/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
08/11/2026 19:51:33 - INFO - httpx - HTTP Request: GET https://huggingface.co/api/models/bert-base-multilingual-cased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
08/11/2026 19:51:33 - INFO - httpx - HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-multilingual-cased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
08/11/2026 19:51:34 - INFO - httpx - HTTP 

Đang khởi tạo model + nạp checkpoint...


08/11/2026 19:51:39 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/bert-base-multilingual-cased/resolve/main/config.json "HTTP/1.1 200 OK"
08/11/2026 19:51:39 - INFO - httpx - HTTP Request: GET https://huggingface.co/bert-base-multilingual-cased/resolve/main/config.json "HTTP/1.1 200 OK"
08/11/2026 19:51:40 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/bert-base-multilingual-cased/resolve/main/model.safetensors "HTTP/1.1 302 Found"
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2862.54it/s]
[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.

Nạp xong [mBERT + Bi-LSTM + CRF]. Checkpoint từ epoch 8, best_f1 lúc lưu = 0.5354


## 3. Hàm suy luận (punctuate)

Giống hệt cách xử lý word-level gather trong `punc_dataset_word.py` lúc train, đảm bảo nhất quán
train/inference.

In [5]:
PUNCT_MAP = {
    "O": "",
    "PERIOD": ".",
    "COMMA": ",",
    "COLON": ":",
    "QMARK": "?",
    "EXCLAM": "!",
    "SEMICOLON": ";",
}
SENTENCE_END_LABELS = {"PERIOD", "QMARK", "EXCLAM"}


def _encode_words_for_inference(words, tokenizer, max_seq_length):
    bos_id = tokenizer.bos_token_id if tokenizer.bos_token_id is not None else tokenizer.cls_token_id
    eos_id = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else tokenizer.sep_token_id
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0

    budget = max_seq_length - 2
    subword_ids, word_starts, kept_words = [], [], []
    truncated = False

    for w in words:
        piece_ids = tokenizer.encode(w, add_special_tokens=False)
        if not piece_ids:
            continue
        if len(subword_ids) + len(piece_ids) > budget:
            truncated = True
            break
        subword_ids.extend(piece_ids)
        word_starts.extend([1] + [0] * (len(piece_ids) - 1))
        kept_words.append(w)

    input_ids = [bos_id] + subword_ids + [eos_id]
    word_starts_full = [0] + word_starts + [0]
    attention_mask = [1] * len(input_ids)

    while len(input_ids) < max_seq_length:
        input_ids.append(pad_id)
        attention_mask.append(0)
        word_starts_full.append(0)

    word_mask = [1] * len(kept_words)
    while len(word_mask) < max_seq_length:
        word_mask.append(0)

    return input_ids, attention_mask, word_starts_full, word_mask, kept_words, truncated


@torch.no_grad()
def punctuate(text: str, capitalize: bool = True) -> str:
    words = text.strip().split()
    if not words:
        return text

    input_ids, attention_mask, word_starts, word_mask, kept_words, truncated = \
        _encode_words_for_inference(words, tokenizer, MAX_SEQ_LENGTH)

    if truncated:
        print(f"[Cảnh báo] Câu dài hơn max_seq_length={MAX_SEQ_LENGTH} subword, "
              f"đã cắt bớt còn {len(kept_words)}/{len(words)} từ.")

    input_ids_t = torch.tensor([input_ids], dtype=torch.long, device=device)
    attention_mask_t = torch.tensor([attention_mask], dtype=torch.long, device=device)
    word_starts_t = torch.tensor([word_starts], dtype=torch.long, device=device)
    word_mask_t = torch.tensor([word_mask], dtype=torch.long, device=device)

    pred_seqs = model(input_ids_t, attention_mask_t, word_starts_t,
                       label_ids=None, word_mask=word_mask_t)
    pred_labels = [ID2LABEL[i] for i in pred_seqs[0][:len(kept_words)]]

    out_tokens = []
    cap_next = capitalize
    for w, lab in zip(kept_words, pred_labels):
        token = w[0].upper() + w[1:] if (cap_next and w) else w
        cap_next = False
        out_tokens.append(token)
        mark = PUNCT_MAP.get(lab, "")
        if mark:
            out_tokens[-1] = out_tokens[-1] + mark
        if lab in SENTENCE_END_LABELS:
            cap_next = capitalize

    return " ".join(out_tokens)


print("Hàm punctuate() đã sẵn sàng.")


Hàm punctuate() đã sẵn sàng.


## 4. Thử suy luận — sửa danh sách câu bên dưới theo ý Fa

In [6]:
SENTENCES = [
    "hôm nay trời đẹp quá chúng ta cùng đi chơi nhé",
    "bạn có khỏe không tôi rất nhớ bạn",
    "xin chào tôi là sinh viên năm cuối nghiên cứu về xử lý ngôn ngữ tự nhiên",
    "anh ơi có cần giúp gì không nếu cần cứ gọi tôi bất cứ lúc nào",
]

for s in SENTENCES:
    result = punctuate(s)
    print("Input :", s)
    print("Output:", result)
    print("-" * 80)


Input : hôm nay trời đẹp quá chúng ta cùng đi chơi nhé
Output: Hôm nay trời đẹp quá, chúng ta cùng đi chơi nhé!
--------------------------------------------------------------------------------
Input : bạn có khỏe không tôi rất nhớ bạn
Output: Bạn có khỏe không? Tôi rất nhớ bạn.
--------------------------------------------------------------------------------
Input : xin chào tôi là sinh viên năm cuối nghiên cứu về xử lý ngôn ngữ tự nhiên
Output: Xin chào tôi, là sinh viên năm cuối nghiên cứu về xử lý ngôn ngữ tự nhiên.
--------------------------------------------------------------------------------
Input : anh ơi có cần giúp gì không nếu cần cứ gọi tôi bất cứ lúc nào
Output: Anh ơi, có cần giúp gì không? Nếu cần cứ gọi tôi bất cứ lúc nào.
--------------------------------------------------------------------------------


## 5. (Tùy chọn) Thử nhanh 1 câu tự nhập

In [7]:
custom_sentence = "xin chào tất cả mọi người ạ tôi tên là âu sao mai hiện là sinh viên năm cuối trường đại học kinh tế thành phố hồ chí minh"
print(punctuate(custom_sentence))


Xin chào tất cả mọi người ạ. Tôi tên là âu sao mai, hiện là sinh viên năm cuối trường đại học kinh tế thành phố hồ chí minh.
